# Retail Expansion & Climate-Resilient Merchandising Case Study

This consolidated notebook merges spatial demographic data, competitive landscapes, and climate vulnerability indexes into a dual-purpose operations and real estate spatial model.


## Phase 1: Environment & Dependency Setup


In [ ]:
import os
import requests
import pandas as pd
import geopandas as gpd
import duckdb
import folium
from folium.plugins import FastMarkerCluster
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display, clear_output
from shapely.geometry import Point
import pygris
import numpy as np

## Phase 2: Programmatic Data Sourcing

Sourcing Census Geometries, VA Open Data Demographics (Population/Income), and USDA Food Access.


In [ ]:
print("Fetching tract geometries via pygris...")
va_tracts = pygris.tracts(state="VA", cb=True, year=2022)

print("Sourcing demographics from Virginia Open Data Portal...")
pop_url = "https://data.virginia.gov/dataset/5b321c48-c444-46e8-a8d5-b815bb252f07/resource/6c5563fd-470e-4e45-8d7b-b56d4e1f6e78/download/acs5_b01003_api_populationatbg.csv"
inc_url = "https://data.virginia.gov/dataset/5a7ab50a-d23f-470a-a443-b8e60b36b8dd/resource/dd1766fb-def6-4929-b37b-338718237eae/download/acs5_b19013_medianhhincomebybg.csv"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
df_demos = None

try:
    import io
    pop_resp = requests.get(pop_url, headers=headers, timeout=30)
    pop_resp.raise_for_status()
    df_pop = pd.read_csv(io.StringIO(pop_resp.text))
    df_pop = df_pop[df_pop['Year'] == 2022]
    
    inc_resp = requests.get(inc_url, headers=headers, timeout=30)
    inc_resp.raise_for_status()
    df_inc = pd.read_csv(io.StringIO(inc_resp.text))
    df_inc = df_inc[df_inc['Year'] == 2022]
    
    m = pd.merge(df_pop, df_inc, on='GEO_ID', suffixes=('_pop', '_inc'))
    m['GEOID'] = m['GEO_ID'].apply(lambda x: x.split('US')[1][:-1])
    m['TotalEstimate_inc'] = m['TotalEstimate_inc'].apply(lambda x: x if x >= 0 else None)
    m['weighted_inc'] = m['TotalEstimate_pop'] * m['TotalEstimate_inc']
    
    tract_pop = m.groupby('GEOID')['TotalEstimate_pop'].sum().reset_index(name='total_population')
    m_valid_inc = m.dropna(subset=['TotalEstimate_inc'])
    tract_weighted_inc = m_valid_inc.groupby('GEOID').apply(
        lambda g: (g['weighted_inc'].sum() / g['TotalEstimate_pop'].sum()) if g['TotalEstimate_pop'].sum() > 0 else None
    ).reset_index(name='median_income')
    
    df_demos = pd.merge(tract_pop, tract_weighted_inc, on='GEOID', how='left')
    state_median_income = df_demos['median_income'].median()
    df_demos['median_income'] = df_demos['median_income'].fillna(state_median_income)
    
    # USDA Food Access data
    usda_csv = "usda_food_access_va.csv"
    df_usda = None
    if os.path.exists(usda_csv):
        df_usda = pd.read_csv(usda_csv)
    else:
        usda_zip_url = "https://www.ers.usda.gov/media/5627/food-access-research-atlas-data-download-2019.zip"
        try:
            import zipfile
            resp = requests.get(usda_zip_url, headers=headers, timeout=40)
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                with z.open("Food Access Research Atlas.csv") as f:
                    df_full = pd.read_csv(f)
                    df_usda = df_full[df_full['State'].astype(str).str.lower().isin(['virginia', 'va'])]
                    df_usda.to_csv(usda_csv, index=False)
        except Exception as ex:
            print(f"Warning: Failed to fetch USDA Food Access data ({ex}).")
            
    if df_usda is not None:
        df_usda_clean = df_usda[['CensusTract', 'LILATracts_1And10']].rename(columns={
            'CensusTract': 'GEOID',
            'LILATracts_1And10': 'food_desert'
        })
        df_usda_clean['GEOID'] = df_usda_clean['GEOID'].astype(str)
        df_usda_clean = df_usda_clean.drop_duplicates(subset=['GEOID'])
        df_demos = pd.merge(df_demos, df_usda_clean, on='GEOID', how='left').fillna({'food_desert': 0})
    else:
        df_demos['food_desert'] = 0.0
        
except Exception as e:
    raise RuntimeError(f"Demo pipeline error: {e}")

### Sourcing FEMA National Risk Index (Heat Wave Variable)


In [ ]:
print("Sourcing FEMA Heat Wave Vulnerability data...")
fema_csv = "fema_heat_wave_va.csv"
df_fema = None

if os.path.exists(fema_csv):
    print(f"Loading FEMA data from cache: {fema_csv}...")
    df_fema = pd.read_csv(fema_csv)
else:
    fema_url = "https://services.arcgis.com/XG15cJAlne2vxtgt/arcgis/rest/services/National_Risk_Index_Census_Tracts/FeatureServer/0/query"
    features = []
    offset = 0
    page_size = 1000
    headers_fema = {
        "Accept-Encoding": "identity",
        "User-Agent": "Mozilla/5.0"
    }
    try:
        while True:
            params = {
                "where": "STATE = 'Virginia'",
                "outFields": "TRACTFIPS,HWAV_RISKS,HWAV_RISKR",
                "resultOffset": offset,
                "resultRecordCount": page_size,
                "f": "json"
            }
            resp = requests.get(fema_url, params=params, headers=headers_fema, timeout=40)
            resp.raise_for_status()
            data = resp.json()
            page_features = data.get("features", [])
            if not page_features:
                break
            features.extend(page_features)
            offset += len(page_features)
            if len(page_features) < page_size:
                break
        
        fema_rows = []
        for f in features:
            attrs = f.get("attributes", {})
            if attrs:
                fema_rows.append({
                    "GEOID": str(attrs.get("TRACTFIPS")),
                    "heat_wave_risk_score": attrs.get("HWAV_RISKS"),
                    "heat_wave_risk_rating": attrs.get("HWAV_RISKR")
                })
        df_fema = pd.DataFrame(fema_rows)
        df_fema = df_fema.drop_duplicates(subset=["GEOID"])
        df_fema.to_csv(fema_csv, index=False)
        print(f"FEMA data saved to cache.")
    except Exception as ex:
        print(f"Warning: Failed to fetch FEMA NRI data ({ex}). Using fallback...")
        
if df_fema is None:
    raise RuntimeError("Failed to fetch FEMA NRI data.")
    
df_fema["GEOID"] = df_fema["GEOID"].astype(str)

# Merge variables and bind onto geometries
df_combined = pd.merge(df_demos, df_fema, on="GEOID", how="left")
median_heat = df_combined["heat_wave_risk_score"].median() or 50.0
df_combined["heat_wave_risk_score"] = df_combined["heat_wave_risk_score"].fillna(median_heat)
df_combined["heat_wave_risk_rating"] = df_combined["heat_wave_risk_rating"].fillna("Relatively Moderate")

va_spatial_demos = va_tracts.merge(df_combined, on='GEOID', how='inner')
print(f"Tract shapes loaded with climate hazards: {len(va_spatial_demos)} shapes.")

### Sourcing Competitors from Overture Maps Foundation (Cloud-Native Parquet via DuckDB)


In [ ]:
print("Querying Overture Maps Foundation (Places) via DuckDB...")
overture_con = duckdb.connect()
overture_con.execute("INSTALL httpfs; LOAD httpfs;")
overture_con.execute("INSTALL spatial; LOAD spatial;")

# Overture Maps Places dataset (Filtering by bounding box and name)
overture_query = '''
SELECT 
    names.primary AS name,
    ST_AsWKB(geometry) AS geom_wkb
FROM read_parquet('s3://overturemaps-us-west-2/release/2026-06-17.0/theme=places/type=*/*', filename=true, hive_partitioning=1)
WHERE bbox.xmin > -84.0 AND bbox.ymin > 36.0 AND bbox.xmax < -75.0 AND bbox.ymax < 40.0
  AND LOWER(names.primary) LIKE '%dollar%'
'''

try:
    df_overture = overture_con.execute(overture_query).df()
    
    import shapely.wkb
    df_overture['geometry'] = df_overture['geom_wkb'].apply(lambda x: shapely.wkb.loads(bytes(x)))
    
    # Filter strictly for Family Dollar vs other Dollar brands
    df_overture['brand'] = df_overture['name']
    df_overture['is_family_dollar'] = df_overture['name'].apply(lambda x: 1 if 'family dollar' in str(x).lower() else 0)
    
    df_overture = df_overture.drop(columns=['geom_wkb'])
    competitor_pts = gpd.GeoDataFrame(df_overture, geometry='geometry', crs="EPSG:4326")
    
    # De-duplicate stores by name within a ~300m radius (shopping center standard)
    competitor_pts_proj = competitor_pts.to_crs("EPSG:3968")
    competitor_pts_proj['x_grid'] = (competitor_pts_proj.geometry.x / 300).round()
    competitor_pts_proj['y_grid'] = (competitor_pts_proj.geometry.y / 300).round()
    competitor_pts = competitor_pts_proj.drop_duplicates(subset=['name', 'x_grid', 'y_grid']).drop(columns=['x_grid', 'y_grid']).to_crs("EPSG:4326")
    
    # Clip explicitly to the Virginia boundary
    va_boundary = va_tracts[['geometry']].to_crs("EPSG:4326")
    competitor_pts = gpd.sjoin(competitor_pts, va_boundary, how="inner", predicate="intersects").drop(columns=['index_right'])
    
    print(f"Competitor points loaded, deduplicated, and clipped to VA: {len(competitor_pts)}")
except Exception as e:
    raise RuntimeError(f"Failed to fetch Overture Maps data: {e}")


## Phase 3: Spatial Joins & DuckDB Spatial SQL

Projecting to Virginia State Plane (EPSG:3968) and performing distance-based counts.


In [ ]:
# Re-project to NAD83 / Virginia State Plane (meters)
va_spatial_demos = va_spatial_demos.to_crs(epsg=3968)
competitor_pts = competitor_pts.to_crs(epsg=3968)

# Expose WKB columns to DuckDB
va_spatial_demos['geom_wkb'] = va_spatial_demos['geometry'].to_wkb()
competitor_pts['geom_wkb'] = competitor_pts['geometry'].to_wkb()

va_spatial_demos_db = pd.DataFrame(va_spatial_demos.drop(columns=['geometry']))
competitor_pts_db = pd.DataFrame(competitor_pts.drop(columns=['geometry']))

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

# Query splits competitors and existing Family Dollars
join_query = '''
SELECT 
    v.GEOID,
    COUNT(CASE WHEN c.is_family_dollar = 0 THEN 1 END) AS comp_count,
    COUNT(CASE WHEN c.is_family_dollar = 1 THEN 1 END) AS fd_count
FROM va_spatial_demos_db v
LEFT JOIN competitor_pts_db c 
  ON ST_Distance(ST_GeomFromWKB(v.geom_wkb), ST_GeomFromWKB(c.geom_wkb)) < 5000
GROUP BY v.GEOID
'''
df_counts = con.execute(join_query).df()
va_spatial_demos = va_spatial_demos.merge(df_counts, on='GEOID', how='left').fillna({'comp_count': 0, 'fd_count': 0})
va_spatial_demos = va_spatial_demos.drop(columns=['geom_wkb'])

# Filter unpopulated areas
va_spatial_demos = va_spatial_demos[va_spatial_demos['total_population'] > 0]

# Perform Min-Max scaling for score consistency
for col in ['total_population', 'median_income', 'comp_count', 'fd_count', 'food_desert', 'heat_wave_risk_score']:
    min_val = va_spatial_demos[col].min()
    max_val = va_spatial_demos[col].max()
    if max_val != min_val:
        va_spatial_demos[f'{col}_norm'] = (va_spatial_demos[col] - min_val) / (max_val - min_val)
    else:
        va_spatial_demos[f'{col}_norm'] = 0.0

print("Spatial joins and scaling completed.")

## Phase 4: Operations / Merchandising Model

Allocating Heat Wave Inventory Tiers for existing Family Dollar stores.


In [ ]:
# Project back to WGS84 for mapping and point-in-polygon check
va_spatial_demos_wgs = va_spatial_demos.to_crs(epsg=4326)
competitor_pts_wgs = competitor_pts.to_crs(epsg=4326)

# Simplify geometry to keep map payload small (tolerance 0.002 degrees ~ 220m)
va_spatial_demos_wgs['geometry'] = va_spatial_demos_wgs['geometry'].simplify(tolerance=0.002, preserve_topology=True)
# Keep only the necessary columns to reduce HTML payload size
cols_to_keep = [
    'GEOID', 'total_population', 'median_income', 'comp_count', 'fd_count',
    'heat_wave_risk_score', 'total_population_norm', 'median_income_norm',
    'comp_count_norm', 'fd_count_norm', 'heat_wave_risk_score_norm',
    'food_desert_norm', 'geometry'
]
va_spatial_demos_wgs = va_spatial_demos_wgs[cols_to_keep].copy()

# Filter Family Dollar stores
fd_stores = competitor_pts_wgs[competitor_pts_wgs['is_family_dollar'] == 1].copy()

# Spatial join overlay: point within polygon
fd_stores_joined = gpd.sjoin(fd_stores, va_spatial_demos_wgs, how="inner", predicate="within")

def get_tier(score):
    if pd.isna(score):
        return "Tier 3: Low Risk"
    if score >= 66.0:
        return "Tier 1: Critical Risk"
    elif score >= 33.0:
        return "Tier 2: Moderate Risk"
    else:
        return "Tier 3: Low Risk"
        
fd_stores_joined['heat_tier'] = fd_stores_joined['heat_wave_risk_score'].apply(get_tier)
print("Operations Merchandising Tiers allocated for existing Family Dollars:")
print(fd_stores_joined['heat_tier'].value_counts())

## Phase 5: Dynamic Visualization Dashboard (Real Estate Suitability)

Interactive WLC model mapping real estate white space.


In [ ]:
# Dedicated output widget
def update_map(pop_weight, income_weight, comp_weight, fd_overlap_penalty, food_desert_weight, climate_weight):
        # Compute dynamic suitability score
        va_spatial_demos_wgs['suitability_score'] = (
            (va_spatial_demos_wgs['total_population_norm'] * pop_weight) +
            (va_spatial_demos_wgs['median_income_norm'] * income_weight) -
            (va_spatial_demos_wgs['comp_count_norm'] * comp_weight) -
            (va_spatial_demos_wgs['fd_count_norm'] * fd_overlap_penalty) +
            (va_spatial_demos_wgs['food_desert_norm'] * food_desert_weight) +
            (va_spatial_demos_wgs['heat_wave_risk_score_norm'] * climate_weight)
        )
        
        vmin = va_spatial_demos_wgs['suitability_score'].min()
        vmax = va_spatial_demos_wgs['suitability_score'].max()
        if vmin == vmax: vmax = vmin + 0.01
        
        colormap = cm.LinearColormap(colors=['#ffffcc', '#fd8d3c', '#e31a1c'], vmin=vmin, vmax=vmax)
        colormap.caption = "Retail Suitability + Climate Risk Score"
        
        m = folium.Map(location=[37.5, -78.5], zoom_start=7, tiles="OpenStreetMap")
        
        style_function = lambda x: {
            'fillColor': colormap(x['properties']['suitability_score']),
            'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7
        }
        
        folium.GeoJson(
            va_spatial_demos_wgs,
            style_function=style_function,
            name="Expansion Suitability",
            tooltip=folium.GeoJsonTooltip(fields=['GEOID', 'suitability_score'], aliases=['Tract ID:', 'Score:'], localize=True, sticky=False),
            popup=folium.GeoJsonPopup(
                fields=['GEOID', 'total_population', 'median_income', 'comp_count', 'fd_count', 'heat_wave_risk_score', 'suitability_score'],
                aliases=['Tract ID:', 'Pop:', 'Med Income:', 'Comps:', 'Sister Stores:', 'Heat Risk:', 'Suitability Score:'],
                localize=True, max_width=300
            )
        ).add_to(m)
        
        # Add Stores Layer
        from folium.plugins import MarkerCluster
        fd_pts = competitor_pts_wgs[competitor_pts_wgs['is_family_dollar'] == 1]
        other_pts = competitor_pts_wgs[competitor_pts_wgs['is_family_dollar'] == 0]
        
        fd_cluster = MarkerCluster(name="Existing Stores", show=False).add_to(m)
        for _, row in fd_pts.iterrows():
            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                popup=folium.Popup(f"<b>Brand:</b> {row['brand']}<br><b>Name:</b> {row['name']}", max_width=250),
                icon=folium.Icon(color='red', icon='home')
            ).add_to(fd_cluster)
            
        comp_cluster = MarkerCluster(name="Competitors", show=False).add_to(m)
        for _, row in other_pts.iterrows():
            folium.Marker(
                location=[row.geometry.y, row.geometry.x],
                popup=folium.Popup(f"<b>Brand:</b> {row['brand']}<br><b>Name:</b> {row['name']}", max_width=250),
                icon=folium.Icon(color='blue', icon='shopping-cart')
            ).add_to(comp_cluster)
        
        colormap.add_to(m)
        folium.LayerControl().add_to(m)
        
        # Using explicit display(m) instead of HTML(_repr_html_())
        # which is more stable in Jupyter widgets and avoids multiple renders on trust.
        display(m)

style = {'description_width': 'initial'}
slider_layout = widgets.Layout(width='95%')

pop_slider = widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='Population Weight', style=style, layout=slider_layout, continuous_update=False)
income_slider = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, description='Income Weight', style=style, layout=slider_layout, continuous_update=False)
comp_slider = widgets.FloatSlider(value=1.5, min=0.0, max=2.0, step=0.1, description='Competition Penalty', style=style, layout=slider_layout, continuous_update=False)
fd_slider = widgets.FloatSlider(value=1.5, min=0.0, max=2.0, step=0.1, description='FD Overlap Penalty', style=style, layout=slider_layout, continuous_update=False)
desert_slider = widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='Food Desert Weight', style=style, layout=slider_layout, continuous_update=False)
climate_slider = widgets.FloatSlider(value=1.2, min=0.0, max=2.0, step=0.1, description='Heat Risk Weight', style=style, layout=slider_layout, continuous_update=False)

col1 = widgets.VBox([pop_slider, income_slider, desert_slider], layout=widgets.Layout(width='50%'))
col2 = widgets.VBox([comp_slider, fd_slider, climate_slider], layout=widgets.Layout(width='50%'))
control_grid = widgets.HBox([col1, col2], layout=widgets.Layout(border='1px solid #ddd', padding='15px', margin='10px 0', border_radius='8px', background_color='#fafafa'))

# Use a dedicated Output widget to prevent multiple maps appending when interacting or reloading
map_output = widgets.Output()

def on_sliders_change(change=None):
    with map_output:
        clear_output(wait=True)
        update_map(
            pop_weight=pop_slider.value,
            income_weight=income_slider.value,
            comp_weight=comp_slider.value,
            fd_overlap_penalty=fd_slider.value,
            food_desert_weight=desert_slider.value,
            climate_weight=climate_slider.value
        )

# Bind observers to the sliders
pop_slider.observe(on_sliders_change, names='value')
income_slider.observe(on_sliders_change, names='value')
comp_slider.observe(on_sliders_change, names='value')
fd_slider.observe(on_sliders_change, names='value')
desert_slider.observe(on_sliders_change, names='value')
climate_slider.observe(on_sliders_change, names='value')

# Trigger initial render
on_sliders_change()

dashboard_ui = widgets.VBox([control_grid, map_output])

display(dashboard_ui)